# RSVP Geometric Diagnostics Notebook

This notebook explores geometric diagnostics for neural representations using the
`rsvp_tools` package:

- Jacobian-based sensitivity and distortion measures
- Local "curvature-like" diagnostics via Jacobian spectra
- Layerwise geometric profiles along a training trajectory

It is designed to be a sandbox for empirical geometry aligned with the monograph's
    theory of representation manifolds, stratification, and lamphrodynamic smoothing.

In [ ]:
import torch
from torch import nn, optim
import matplotlib.pyplot as plt
import numpy as np

from rsvp_tools import StratumIndexEstimator, StratumIndexConfig
from rsvp_tools.utils import jacobian_vp, approximate_rank

torch.manual_seed(0)
print('Imports successful.')

## Define a Simple Representation Model

We treat the penultimate layer as a representation map \(\Phi: \mathbb{R}^{d_{in}} \to \mathbb{R}^{d_{rep}}\).

In [ ]:
class RepNet(nn.Module):
    def __init__(self, d_in=16, d_hidden=32, d_rep=8, d_out=1):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_rep)
        self.fc3 = nn.Linear(d_rep, d_out)

    def representation(self, x):
        h = torch.relu(self.fc1(x))
        z = torch.relu(self.fc2(h))
        return z

    def forward(self, x):
        z = self.representation(x)
        return self.fc3(z)

model = RepNet()
print(model)

## Synthetic Data

We create a structured regression task so that the representation manifold is nontrivial.

In [ ]:
def sample_data(n=512, d_in=16):
    x = torch.randn(n, d_in)
    # Nonlinear target: sum of squares on a subset of coordinates
    y = (x[:, :4]**2).sum(dim=1, keepdim=True) + 0.1 * torch.randn(n, 1)
    return x, y

x_train, y_train = sample_data()
x_train.shape, y_train.shape

## Train the Model Briefly

We do a light optimization run to obtain nontrivial geometry in the learned representation.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

loss_history = []
for epoch in range(50):
    optimizer.zero_grad()
    pred = model(x_train)
    loss = criterion(pred, y_train)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

plt.figure(figsize=(6,4))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

## Local Jacobian Diagnostics

We examine the Jacobian of the representation map with respect to inputs,
and compute norms and spectra to probe local distortion and curvature-like behavior.

In [ ]:
def representation_jacobian(model, x_batch):
    """Compute a dense Jacobian of representation wrt inputs for small batches."""
    x_batch = x_batch.clone().detach().requires_grad_(True)
    z = model.representation(x_batch)  # (B, d_rep)
    B, d_rep = z.shape
    d_in = x_batch.shape[1]

    J = torch.zeros(B, d_rep, d_in)
    for i in range(d_rep):
        grads = torch.autograd.grad(z[:, i].sum(), x_batch, retain_graph=True)[0]
        J[:, i, :] = grads
    return J

batch = x_train[:64]
J = representation_jacobian(model, batch)
J.shape

### Jacobian Norms and Local Distortion

We compute Frobenius norms of the Jacobian for each sample as a proxy for local sensitivity.

In [ ]:
J_norms = torch.linalg.norm(J, dim=(1,2))  # per-sample

plt.figure(figsize=(6,4))
plt.hist(J_norms.detach().numpy(), bins=20)
plt.xlabel('‖J‖_F (per-sample)')
plt.ylabel('Count')
plt.title('Distribution of Representation Jacobian Norms')
plt.show()

### Local Metric Spectra

At each sample, we approximate the pullback metric via \(G = J^T J\) and inspect its eigenvalues.
These serve as curvature-like descriptors of anisotropy and local capacity.

In [ ]:
eigvals_all = []
for b in range(J.shape[0]):
    G = J[b].T @ J[b]  # (d_in, d_in)
    # Symmetric positive semi-definite; use eigvalsh for numerical stability
    eigvals = torch.linalg.eigvalsh(G)
    eigvals_all.append(eigvals.detach().numpy())

eigvals_all = np.stack(eigvals_all, axis=0)  # (B, d_in)

plt.figure(figsize=(6,4))
plt.plot(np.sort(eigvals_all.mean(axis=0))[::-1])
plt.xlabel('Index (sorted)')
plt.ylabel('Mean eigenvalue of G')
plt.title('Average Metric Spectrum (J^T J)')
plt.show()

## Stratum Index as a Global Geometric Diagnostic

We connect local geometric diagnostics to the global Stratum Index.

In [ ]:
si_config = StratumIndexConfig(num_samples=6, batch_size=32)
si_estimator = StratumIndexEstimator(si_config)

def input_sampler(batch_size):
    x, _ = sample_data(batch_size)
    return x

si_value = si_estimator.estimate(model, input_sampler)
si_value

We can also track the Stratum Index over a short training run to observe its
relationship to loss and Jacobian norms.

In [ ]:
model2 = RepNet()
criterion2 = nn.MSELoss()
opt2 = optim.Adam(model2.parameters(), lr=1e-3)

si_traj = []
loss_traj = []
norm_traj = []

for epoch in range(20):
    opt2.zero_grad()
    pred = model2(x_train)
    loss = criterion2(pred, y_train)
    loss.backward()
    opt2.step()

    # Stratum Index
    si = si_estimator.estimate(model2, input_sampler)
    si_traj.append(si)
    loss_traj.append(loss.item())

    # Mean Jacobian norm on a small batch
    J_local = representation_jacobian(model2, x_train[:64])
    norm_traj.append(torch.linalg.norm(J_local, dim=(1,2)).mean().item())

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.plot(loss_traj)
plt.title('Loss')
plt.xlabel('Epoch')

plt.subplot(1,3,2)
plt.plot(si_traj)
plt.title('Stratum Index')
plt.xlabel('Epoch')

plt.subplot(1,3,3)
plt.plot(norm_traj)
plt.title('Mean ‖J‖_F')
plt.xlabel('Epoch')

plt.tight_layout()
plt.show()